# S2D Initialization Consistency And Drift Diagnostics

Unified workflow for diagnosing initialization shock, coupled consistency, lead-time drift, and prediction skill in S2D hindcasts.

This notebook is organized around one user control panel, then reusable diagnostics:

1. Load/cache regional hindcast indices for one or more experiments and variables.
2. Load matching observed regional monthly time series where an observation product is configured.
3. Compute lead-time diagnostics: ensemble mean, ensemble spread, model-minus-obs error, climatological bias, early shock index, month-to-month drift rate, RMSE, and ACC.
4. Compute coupled consistency diagnostics such as `SST_minus_T2m` and `net_surface_heat_flux`.
5. Save figures to `FIGURE_OUTDIR` using a consistent `fig_<variable_or_diagnostic>_<model>_<reference>_<period>_<other>.png` naming style.

The diagnostics are computed directly from regional monthly indices, so the workflow works for short exploratory windows as well as longer climatology periods.


In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
from pathlib import Path
import re
import itertools as _itertools

import cftime
import dask
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from esp_lab import data_access_e3sm as data_access
from esp_lab import data_access_obs as obs_access
from esp_lab import stats
from esp_lab.utils import spatial_utils as spatial
from esp_lab.utils import calendar_utils as cal
from esp_lab.utils import mov_utils as mov
from esp_lab.utils.dask_utils import DaskConfig, get_cluster_client, close_cluster
from esp_lab.diagnostics import S2DDiagnostics, S2DConfig

FIGURE_OUTDIR = Path("/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag")
FIGURE_OUTDIR.mkdir(parents=True, exist_ok=True)


def figure_filename(*parts, ext="png"):
    """Build readable figure filenames: fig_<diagnostic>_<model>_<reference>_<period>_<other>.ext."""
    clean_parts = ["fig"]
    for part in parts:
        if part is None:
            continue
        text = str(part).strip()
        if not text:
            continue
        text = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_").lower()
        if text:
            clean_parts.append(text)
    return "_".join(clean_parts) + f".{ext.lstrip('.').lower()}"


def convert_kelvin_to_celsius(da):
    out = da - 273.15
    out.attrs.update(da.attrs)
    out.attrs["units"] = "degC"
    return out


def _temperature_units_kind(da):
    units = str(da.attrs.get("units", "")).strip().lower()
    if units in {"k", "kelvin", "degrees_k", "degree_k"} or "kelvin" in units:
        return "kelvin"
    if any(token in units for token in ("degc", "degree_c", "degrees_c", "celsius", "c")):
        return "celsius"
    return "unknown"


def convert_kelvin_to_celsius_if_needed(da):
    kind = _temperature_units_kind(da)
    if kind == "kelvin":
        return convert_kelvin_to_celsius(da)
    if kind == "celsius":
        return da

    # Fallback for regional/cache series with missing units.
    try:
        indexers = {dim: 0 for dim in da.dims if da.sizes.get(dim, 0) > 0}
        sample = float(da.isel(indexers).mean(skipna=True).load())
    except Exception:
        return da
    if np.isfinite(sample) and sample > 100.0:
        return convert_kelvin_to_celsius(da)
    return da


def normalize_regional_temperature_units(da):
    kind = _temperature_units_kind(da)
    if kind == "kelvin":
        return convert_kelvin_to_celsius(da)
    if kind == "celsius":
        return da
    try:
        mean_value = float(da.mean(skipna=True).load())
    except Exception:
        return da
    if np.isfinite(mean_value) and mean_value > 100.0:
        return convert_kelvin_to_celsius(da)
    return da


def convert_precip_mps_to_mmday(da):
    out = da * 1000.0 * 86400.0
    out.attrs.update(da.attrs)
    out.attrs["units"] = "mm/day"
    return out


def convert_pa_to_hpa(da):
    out = da * 1.0e-2
    out.attrs.update(da.attrs)
    out.attrs["units"] = "hPa"
    return out


def no_conversion(da):
    return da


Warning 3: Cannot find header.dxf (GDAL_DATA is not defined)
ERROR 1: PROJ: proj_create_from_database: Cannot find proj.db
Warning 3: Cannot find header.dxf (GDAL_DATA is not defined)


In [2]:
print("xarray:", xr.__version__)
print("dask:", dask.__version__)


xarray: 2026.4.0
dask: 2026.3.0


In [3]:
# -----------------------------
# Dask setup
# -----------------------------
machine_env = os.environ.get("CLUSTER_TYPE", "local")

dask_cfg = DaskConfig(
    cluster_type=machine_env,
    workers=30,
    cores=4,
    memory="16GB",
    walltime="02:00:00",
    queue=None,
    project=None,
)

cluster, client = get_cluster_client(dask_cfg)
print(client)


<Client: 'tcp://127.0.0.1:33721' processes=30 threads=30, memory=502.97 GiB>


## User Control Panel


In [4]:
# =========================================================
# USER CONTROL PANEL
# =========================================================
# -----------------------------
# Diagnostic setup knobs
# -----------------------------
# Native E3SM variable used by the original drift plots. Use "SST" as a friendly alias for E3SM "TS".
primary_field = "PRECT"
field = primary_field  # Backward-compatible alias used by the rest of the notebook.

# Extra variables to cache and analyze alongside the primary field.
consistency_fields = ["SST", "TREFHT", "LHFLX", "SHFLX", "FSNS", "FLNS", "PSL", "TAUX", "TAUY"]
surface_plot_fields = ["PRECT", "SST"]
summary_heatmap_companion_fields = ["PRECT", "SST"]
final_plot_fields = ["PRECT", "SST"]
comparison_fields = {"precip": "PRECT", "sst": "SST"}
make_precip_sst_comparison = True
fields = list(dict.fromkeys([field] + consistency_fields + surface_plot_fields + list(comparison_fields.values())))

# Lead windows are 1-based model leads. Lead-month display is 0-based for user-facing plots.
shock_leads = slice(1, 3)
shock_leads_season1 = slice(1, 6)
skill_leads = slice(1, 6)
spinup_lead_months = (0, 2)
spinup_span = (spinup_lead_months[0] - 0.5, spinup_lead_months[1] + 0.5)
spinup_label = "Spin-up window"
lead_label_every = 3
comparison_scatter_label_leads = [1, 4, 7, 13, 19, 24]
experiment_markers = {"JRA55_FOSIRL": "o", "BruteForce": "s"}
show_case_summary_table = True
show_lead_detail_table = False
make_consistency_field_figures = True
quicklook_bar_label_inside_pad = 0.05
write_summary_text_files = True
summary_float_format = "%.4f"

case_nens = 10
case_nlead = 24
members = [f"EN{i:02d}" for i in range(case_nens)]

data_dir = "/global/cfs/cdirs/e3sm/S2S2D/post_process"
outdir = Path("/global/cfs/cdirs/e3smdata/simulations/S2S2D/s2d_diag/leadtime_drift")
outdir.mkdir(parents=True, exist_ok=True)
summary_table_outdir = outdir / "summary_tables"

experiments = {
    "JRA55_FOSIRL": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL",
    "BruteForce": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce",
}

selected_years = [1980, 1981, 1982, 1983, 1984, 1985, 1986]
init_months = [5, 11]
freq_tag = "mon"  # "mon" is recommended for lead-time drift diagnostics.

region = [-170.0, -120.0, -5.0, 5.0]
region_name = "Nino3.4"
reference_name = "obs"

realm = "atm"
grid = "180x360_aave"
freq = "monthly"
ts_split = "2yr"
engine = "netcdf4"
chunks_open = {}
reg_chunk = {"Y": 1, "L": case_nlead, "M": 2}

force_rewrite = False
persist_intermediate = False
load_regional_series = True
convert_ts_to_degC = True
require_all_members = True
verify_field_name = True
verify_coverage = True

# Observation settings and unit conversions. Diagnostics without obs_product still support model-only consistency metrics.
MODEL_FIELD_ALIASES = {"SST": "TS"}
VARIABLE_CONFIG = {
    "SST": {
        "native_field": "TS",
        "plot_name": "SST",
        "plot_units": "degC",
        "obs_product": "HadISST2",
        "obs_var": "sst",
        "obs_yrs": 1979,
        "obs_yre": 2022,
        "model_convert": convert_kelvin_to_celsius_if_needed,  # S2DDiagnostics usually converts TS to degC; this also protects stale Kelvin caches.
        "obs_convert": convert_kelvin_to_celsius_if_needed,
    },
    "TS": {
        "native_field": "TS",
        "plot_name": "SST",
        "plot_units": "degC",
        "obs_product": "HadISST2",
        "obs_var": "sst",
        "obs_yrs": 1979,
        "obs_yre": 2022,
        "model_convert": convert_kelvin_to_celsius_if_needed,
        "obs_convert": convert_kelvin_to_celsius_if_needed,
    },
    "TREFHT": {
        "native_field": "TREFHT",
        "plot_name": "2-m air temperature",
        "plot_units": "degC",
        "obs_product": "ERA5",
        "obs_var": "tas",
        "obs_yrs": 1979,
        "obs_yre": 2019,
        "model_convert": convert_kelvin_to_celsius_if_needed,
        "obs_convert": convert_kelvin_to_celsius_if_needed,
    },
    "PRECT": {
        "native_field": "PRECT",
        "plot_name": "precipitation",
        "plot_units": "mm/day",
        "obs_product": "GPCP_v2.3",
        "obs_var": "pr",
        "obs_yrs": 1979,
        "obs_yre": 2017,
        "model_convert": convert_precip_mps_to_mmday,
        "obs_convert": no_conversion,
    },
    "PSL": {
        "native_field": "PSL",
        "plot_name": "sea-level pressure",
        "plot_units": "hPa",
        "obs_product": "ERA5",
        "obs_var": "psl",
        "obs_yrs": 1979,
        "obs_yre": 2019,
        "model_convert": convert_pa_to_hpa,
        "obs_convert": convert_pa_to_hpa,
    },
    "LHFLX": {"native_field": "LHFLX", "plot_name": "latent heat flux", "plot_units": "W/m2", "obs_product": None, "model_convert": no_conversion},
    "SHFLX": {"native_field": "SHFLX", "plot_name": "sensible heat flux", "plot_units": "W/m2", "obs_product": None, "model_convert": no_conversion},
    "FSNS": {"native_field": "FSNS", "plot_name": "net shortwave surface flux", "plot_units": "W/m2", "obs_product": None, "model_convert": no_conversion},
    "FLNS": {"native_field": "FLNS", "plot_name": "net longwave surface flux", "plot_units": "W/m2", "obs_product": None, "model_convert": no_conversion},
    "TAUX": {"native_field": "TAUX", "plot_name": "zonal surface stress", "plot_units": "N/m2", "obs_product": None, "model_convert": no_conversion},
    "TAUY": {"native_field": "TAUY", "plot_name": "meridional surface stress", "plot_units": "N/m2", "obs_product": None, "model_convert": no_conversion},
}

for requested_field in fields:
    if requested_field not in VARIABLE_CONFIG:
        raise ValueError(f"Unsupported field={requested_field!r}; available={list(VARIABLE_CONFIG)}")

var_cfg = VARIABLE_CONFIG[field]
native_field = var_cfg.get("native_field", MODEL_FIELD_ALIASES.get(field, field))
plot_name = var_cfg["plot_name"]
plot_units = var_cfg["plot_units"]

obs_dir = "/global/cfs/cdirs/e3sm/e3sm_diags/obs_for_e3sm_diags/time-series"
obs_product = var_cfg.get("obs_product")
obs_var = var_cfg.get("obs_var")
obs_yrs = var_cfg.get("obs_yrs")
obs_yre = var_cfg.get("obs_yre")

print("field:", field, "native:", native_field)
print("cached/diagnostic fields:", fields)
print("surface plot fields:", surface_plot_fields)
print("summary heatmap companion fields:", summary_heatmap_companion_fields)
print("final plot fields:", final_plot_fields)
print("comparison fields:", comparison_fields)
print("lead label every:", lead_label_every)
print("scatter label leads:", comparison_scatter_label_leads)
print("show lead detail table:", show_lead_detail_table)
print("make consistency field figures:", make_consistency_field_figures)
print("write summary text files:", write_summary_text_files)
print("summary table output:", summary_table_outdir)
print("spin-up lead months:", spinup_lead_months)
print("experiments:", list(experiments))
print("selected_years:", selected_years)
print("init_months:", init_months)
print("figure output:", FIGURE_OUTDIR)


field: PRECT native: PRECT
cached/diagnostic fields: ['PRECT', 'SST', 'TREFHT', 'LHFLX', 'SHFLX', 'FSNS', 'FLNS', 'PSL', 'TAUX', 'TAUY']
surface plot fields: ['PRECT', 'SST']
summary heatmap companion fields: ['PRECT', 'SST']
final plot fields: ['PRECT', 'SST']
comparison fields: {'precip': 'PRECT', 'sst': 'SST'}
lead label every: 3
scatter label leads: [1, 4, 7, 13, 19, 24]
show lead detail table: False
make consistency field figures: True
write summary text files: True
summary table output: /global/cfs/cdirs/e3smdata/simulations/S2S2D/s2d_diag/leadtime_drift/summary_tables
spin-up lead months: (0, 2)
experiments: ['JRA55_FOSIRL', 'BruteForce']
selected_years: [1980, 1981, 1982, 1983, 1984, 1985, 1986]
init_months: [5, 11]
figure output: /global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag


## Helper Functions


In [5]:
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
init_labels = {2: "FEB init", 5: "MAY init", 8: "AUG init", 11: "NOV init"}
_base_colors = ["royalblue", "firebrick", "forestgreen", "darkorange", "mediumpurple", "saddlebrown", "deeppink", "teal"]
exp_colors = {name: color for name, color in zip(experiments, _itertools.cycle(_base_colors))}

# -----------------------------
# Figure style setup
# -----------------------------
# Base font size in points. Change this one value to scale figure text consistently.
fontz = 14

FS = {
    "suptitle": int(round(fontz * 1.1)),
    "title": int(round(fontz * 1.0)),
    "label": int(round(fontz * 0.95)),
    "tick": int(round(fontz * 0.95)),
    "legend": int(round(fontz * 0.90)),
    "legend_title": int(round(fontz * 0.95)),
    "colorbar": int(round(fontz * 0.95)),
    "table": int(round(fontz * 0.90)),
}

FIG = {
    "dpi": 220,
    "line_width": 2.0,
    "thin_line_width": 0.8,
    "marker_size": 5.0,
    "scatter_size": 42,
    "scatter_edge_width": 0.7,
    "grid_alpha": 0.4,
    "band_alpha": 0.2,
    "line_alpha": 0.55,
    "single_case_figsize": (10.5, 8.2),
    "diagnostic_col_width": 7.0,
    "diagnostic_fig_height": 11.0,
    "season_col_width": 7.0,
    "season_fig_height": 6.0,
    "summary_min_width": 12.0,
    "summary_lead_width": 0.48,
    "summary_row_height": 1.0,
    "summary_extra_height": 3.5,
}

plt.rcParams.update(
    {
        "font.size": FS["label"],
        "axes.titlesize": FS["title"],
        "axes.labelsize": FS["label"],
        "xtick.labelsize": FS["tick"],
        "ytick.labelsize": FS["tick"],
        "legend.fontsize": FS["legend"],
        "figure.titlesize": FS["suptitle"],
    }
)


def apply_axis_style(ax):
    ax.grid(True, alpha=FIG["grid_alpha"])
    ax.tick_params(labelsize=FS["tick"])



def safe_token(value):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_")


def lead_to_cal_month(init_month: int, lead_1based: int) -> int:
    return ((init_month - 1 + int(lead_1based) - 1) % 12) + 1


def lead_to_valid_time(init_year: int, init_month: int, lead_1based: int):
    month_index = init_month - 1 + int(lead_1based) - 1
    year = init_year + month_index // 12
    month = month_index % 12 + 1
    return cftime.DatetimeNoLeap(year, month, 15)


def valid_times_for_leads(init_year: int, init_month: int, leads):
    return [lead_to_valid_time(init_year, init_month, L) for L in leads]


def format_lead_time_axis(ax, leads, init_month, show_labels=True, label_every=None):
    if label_every is None:
        label_every = lead_label_every
    lead_months = np.asarray(leads) - 1
    if len(lead_months) == 0:
        return lead_months
    tick_idx = list(range(0, len(lead_months), label_every))
    if tick_idx[-1] != len(lead_months) - 1:
        tick_idx.append(len(lead_months) - 1)
    major_ticks = lead_months[tick_idx]
    major_labels = [
        f"L{int(lead_months[i])}\n{month_names[lead_to_cal_month(init_month, leads[i]) - 1]}"
        for i in tick_idx
    ]
    ax.set_xlim(float(np.nanmin(lead_months)) - 0.75, float(np.nanmax(lead_months)) + 0.75)
    ax.set_xticks(major_ticks)
    ax.set_xticks(lead_months, minor=True)
    if show_labels:
        ax.set_xticklabels(major_labels, fontsize=FS["tick"])
    else:
        ax.tick_params(axis="x", labelbottom=False)
    ax.grid(which="minor", axis="x", alpha=0.16, linewidth=FIG["thin_line_width"])
    return lead_months


def select_init_year(da, init_year):
    yvals = da.Y.values
    if init_year in yvals:
        return da.sel(Y=init_year)
    if str(init_year) in yvals:
        return da.sel(Y=str(init_year))
    ystr = np.array([str(y) for y in yvals])
    matches = [y for y, ys in zip(yvals, ystr) if ys.startswith(str(init_year))]
    if len(matches) == 1:
        return da.sel(Y=matches[0])
    raise KeyError(f"Cannot match init_year={init_year} in Y: {yvals}")


def obs_clim_by_lead(obs_monthly_clim, init_month: int, leads):
    months = [lead_to_cal_month(init_month, L) for L in leads]
    return xr.DataArray(
        [obs_monthly_clim.sel(month=m).item() for m in months],
        dims=("L",),
        coords={"L": leads},
        name="obs_clim_by_lead",
    )


def obs_values_by_valid_time(obs_series, init_year: int, init_month: int, leads):
    times = valid_times_for_leads(init_year, init_month, leads)
    values = obs_series.sel(time=times, method="nearest").load()
    return xr.DataArray(values.values, dims=("L",), coords={"L": leads}, name="obs_by_lead")


def obs_matrix_by_valid_times(obs_series, init_years, init_month: int, leads):
    rows = []
    for year in init_years:
        times = [lead_to_valid_time(year, init_month, L) for L in leads]
        obs_at_lead = obs_series.sel(time=times, method="nearest").load()
        rows.append(obs_at_lead.values)
    return xr.DataArray(
        np.asarray(rows),
        dims=("Y", "L"),
        coords={"Y": init_years, "L": leads},
        name="obs_valid_by_year_lead",
    )


def obs_mean_by_valid_times(obs_series, init_years, init_month: int, leads):
    return obs_matrix_by_valid_times(obs_series, init_years, init_month, leads).mean("Y", skipna=True).rename("obs_valid_lead_mean")


def infer_model_init_years(da, fallback_years):
    """Infer integer initialization years from a model regional-index Y coordinate."""
    n_years = da.sizes.get("Y")
    inferred = []
    for yval in da.Y.values:
        if hasattr(yval, "year"):
            inferred.append(int(yval.year))
            continue
        text = str(yval)
        match = re.search(r"(\d{4})", text)
        if match:
            inferred.append(int(match.group(1)))
    if len(inferred) == n_years:
        return inferred

    fallback = list(fallback_years)
    if len(fallback) >= n_years:
        return fallback[:n_years]
    raise ValueError(
        "Cannot infer enough model initialization years from da.Y and fallback_years "
        f"({len(fallback)} fallback years for {n_years} model years)."
    )


def compute_rmse_and_acc(model, obs_by_year_lead):
    model_ens_mean = model.mean("M", skipna=True)
    if obs_by_year_lead.sizes.get("Y") != model_ens_mean.sizes.get("Y"):
        raise ValueError(
            "obs_by_year_lead and model must have the same number of initialization years "
            f"({obs_by_year_lead.sizes.get('Y')} vs {model_ens_mean.sizes.get('Y')})."
        )
    obs_aligned = xr.DataArray(
        obs_by_year_lead.values,
        dims=("Y", "L"),
        coords={"Y": model_ens_mean.Y.values, "L": model_ens_mean.L.values},
        name="obs_valid_by_year_lead",
    )
    error = model_ens_mean - obs_aligned
    rmse = np.sqrt((error ** 2).mean("Y", skipna=True)).rename("rmse")

    model_anom = model_ens_mean - model_ens_mean.mean("Y", skipna=True)
    obs_anom = obs_aligned - obs_aligned.mean("Y", skipna=True)
    acc = xr.corr(model_anom, obs_anom, dim="Y").rename("acc")
    return rmse.load(), acc.load()


def safe_sel_mean(da, lead_slice):
    return float(da.sel(L=lead_slice).mean("L", skipna=True))


def shock_index_from_bias(bias, lead_slice):
    return float(np.sqrt((bias.sel(L=lead_slice) ** 2).mean("L", skipna=True)))


def season1_drift_from_bias(bias, lead_slice=slice(1, 6)):
    early = bias.sel(L=lead_slice)
    if early.sizes.get("L", 0) < 2:
        return np.nan
    return float((early.isel(L=-1) - early.isel(L=0)) / (early.L.isel(L=-1) - early.L.isel(L=0)))


def field_cache_token(field_key):
    return safe_token(field_key)


def regional_cache_path(exp_name, init_month, freq_tag, field_key=None):
    cache_field = field_cache_token(field_key or field)
    return outdir / safe_token(exp_name) / f"{safe_token(exp_name)}_init{init_month:02d}_{cache_field}_{safe_token(region_name)}_{freq_tag}.nc"


## Compute Or Load Regional Hindcast Indices


In [6]:
%%time
index_by_field = {}
metadata_by_field = {}

for field_key in fields:
    cfg_for_field = VARIABLE_CONFIG[field_key]
    native_field_key = cfg_for_field.get("native_field", MODEL_FIELD_ALIASES.get(field_key, field_key))
    index_by_exp = {}
    metadata_by_exp = {}

    common_cfg = dict(
        field=native_field_key,
        data_dir=data_dir,
        members=members,
        nlead=case_nlead,
        init_months=init_months,
        init_years={m: selected_years for m in init_months},
        region=region,
        region_name=region_name,
        climy0=min(selected_years),
        climy1=max(selected_years),
        outdir=str(outdir),
        force_rewrite=force_rewrite,
        realm=realm,
        grid=grid,
        freq=freq,
        ts_split=ts_split,
        engine=engine,
        chunks=chunks_open,
        require_all_members=require_all_members,
        verify_field_name=verify_field_name,
        verify_coverage=verify_coverage,
        persist_intermediate=persist_intermediate,
        load_regional_series=load_regional_series,
        convert_ts_to_degC=convert_ts_to_degC,
    )

    for exp_name, case_prefix in experiments.items():
        print("\n" + "=" * 80)
        print(f"Field: {field_key} (native {native_field_key}) | Experiment: {exp_name}")
        exp_dir = outdir / safe_token(exp_name)
        exp_dir.mkdir(parents=True, exist_ok=True)

        index_by_exp[exp_name] = {}
        metadata_by_exp[exp_name] = {}

        needed = [regional_cache_path(exp_name, m, tag, field_key=field_key) for m in init_months for tag in ("mon", "seas")]
        use_cache = all(path.exists() for path in needed) and not force_rewrite

        if use_cache:
            print("[READ] using cached regional files")
            results = None
        else:
            print("[RUN] computing regional indices")
            cfg_exp = S2DConfig(case_prefix=case_prefix, **common_cfg)
            diag_exp = S2DDiagnostics(
                cfg=cfg_exp,
                data_access=data_access,
                obs_access=obs_access,
                stats=stats,
                spatial=spatial,
                cal=cal,
            )
            results = diag_exp.run()

            for init_month in init_months:
                for tag in ("mon", "seas"):
                    da = results[init_month][tag].chunk(reg_chunk)
                    da = cfg_for_field["model_convert"](da).rename("regional_index")
                    da.attrs.update(
                        experiment=exp_name,
                        case_prefix=case_prefix,
                        region=region_name,
                        lonlat=str(region),
                        field=field_key,
                        native_field=native_field_key,
                        units=cfg_for_field["plot_units"],
                        frequency=tag,
                    )
                    path = regional_cache_path(exp_name, init_month, tag, field_key=field_key)
                    if path.exists():
                        path.unlink()
                    da.to_netcdf(path, encoding={"regional_index": {"zlib": True, "complevel": 1}})

        for init_month in init_months:
            index_by_exp[exp_name][init_month] = {}
            for tag in ("mon", "seas"):
                path = regional_cache_path(exp_name, init_month, tag, field_key=field_key)
                da_cached = xr.open_dataarray(path, chunks={}).chunk(reg_chunk)
                if field_key in {"SST", "TS", "TREFHT"}:
                    da_cached = normalize_regional_temperature_units(da_cached)
                    da_cached.attrs["units"] = "degC"
                index_by_exp[exp_name][init_month][tag] = da_cached
                print(field_key, exp_name, init_month, tag, index_by_exp[exp_name][init_month][tag].shape, index_by_exp[exp_name][init_month][tag].attrs.get("units", ""))

    index_by_field[field_key] = index_by_exp
    metadata_by_field[field_key] = metadata_by_exp

# Backward-compatible alias for the primary field used by the existing plots.
index_by_exp = index_by_field[field]
metadata_by_exp = metadata_by_field[field]



Field: PRECT (native PRECT) | Experiment: JRA55_FOSIRL
[READ] using cached regional files
PRECT JRA55_FOSIRL 5 mon (4, 24, 10) mm/day
PRECT JRA55_FOSIRL 5 seas (4, 8, 10) mm/day
PRECT JRA55_FOSIRL 11 mon (4, 24, 10) mm/day
PRECT JRA55_FOSIRL 11 seas (4, 8, 10) mm/day

Field: PRECT (native PRECT) | Experiment: BruteForce
[READ] using cached regional files
PRECT BruteForce 5 mon (4, 24, 10) mm/day
PRECT BruteForce 5 seas (4, 8, 10) mm/day
PRECT BruteForce 11 mon (4, 24, 10) mm/day
PRECT BruteForce 11 seas (4, 8, 10) mm/day

Field: SST (native TS) | Experiment: JRA55_FOSIRL
[READ] using cached regional files
SST JRA55_FOSIRL 5 mon (4, 24, 10) degC
SST JRA55_FOSIRL 5 seas (4, 8, 10) degC
SST JRA55_FOSIRL 11 mon (4, 24, 10) degC
SST JRA55_FOSIRL 11 seas (4, 8, 10) degC

Field: SST (native TS) | Experiment: BruteForce
[READ] using cached regional files
SST BruteForce 5 mon (4, 24, 10) degC
SST BruteForce 5 seas (4, 8, 10) degC
SST BruteForce 11 mon (4, 24, 10) degC
SST BruteForce 11 seas (4

## Load Observed Regional Reference Series


In [7]:
%%time
if obs_product is None:
    obs_mon_regional = None
    obs_mon_clim = None
    obs_regional_for_diag = None
    print(f"No observation product configured for {field}; skipping obs-based diagnostics.")
else:
    obs_chunk = {"time": 120, "lat": 90, "lon": 180}
    lat_name = "lat"
    lon_name = "lon"

    obs_ds = obs_access.get_monthly_data(
        obs_dir=obs_dir,
        field=native_field,
        field_map={native_field: obs_var},
        product=obs_product,
        start_year=obs_yrs,
        end_year=obs_yre,
        chunks=obs_chunk,
        verbose=True,
    )

    obs_da = var_cfg["obs_convert"](obs_ds[obs_var])
    obs_ds[obs_var] = obs_da

    obs_sel_start = min(selected_years)
    obs_sel_end = min(obs_yre, max(selected_years) + 3)
    obs_ds = obs_ds.sel(time=slice(str(obs_sel_start), str(obs_sel_end)))

    landmask_obs = spatial.create_land_sea_mask(obs_ds[obs_var], lat_key=lat_name, lon_key=lon_name).astype(bool)
    oceanmask_obs = (~landmask_obs).chunk({lat_name: 90, lon_name: 180})
    obs_weights = obs_access.obs_regional_weights(
        obs_ds[obs_var],
        region,
        lat_name=lat_name,
        lon_name=lon_name,
        mask=oceanmask_obs,
    )

    obs_mon_regional = obs_ds[obs_var].weighted(obs_weights).mean((lat_name, lon_name)).load()
    obs_mon_regional.name = "obs_regional_index"
    obs_mon_regional.attrs["units"] = plot_units
    obs_mon_clim = obs_mon_regional.groupby("time.month").mean("time")

    if freq_tag == "mon":
        obs_regional_for_diag = obs_mon_regional
    elif freq_tag == "seas":
        obs_regional_for_diag = (
            obs_mon_regional.rolling(time=3, min_periods=3, center=True)
            .mean()
            .dropna("time", how="all")
        )
        obs_regional_for_diag.name = "obs_regional_index_seasonal"
        obs_regional_for_diag.attrs["units"] = plot_units
    else:
        raise ValueError(f"Unsupported freq_tag={freq_tag!r}; expected 'mon' or 'seas'.")

    print(obs_mon_regional)
    print("Obs series used for diagnostics:", obs_regional_for_diag)
    print("Obs monthly climatology:", obs_mon_clim.values)


[OBS] Mapping field 'PRECT' -> 'pr'
[OBS] Loading: /global/cfs/cdirs/e3sm/diagnostics/observations/Atm/time-series/GPCP_v2.3/PRECT_197901_201712.nc
[OBS] File matched field 'PRECT' for requested field 'pr'
CPU times: user 263 ms, sys: 286 ms, total: 549 ms
Wall time: 2.08 s


ImportError: regionmask is required to build the default land mask source.

## Build Lead-Time Diagnostic Tables


In [ ]:
diagnostics = {}

if obs_regional_for_diag is None:
    raise ValueError(f"Primary field {field} does not have observations configured; choose a field with obs_product for bias/skill diagnostics.")

for exp_name in experiments:
    diagnostics[exp_name] = {}
    for init_month in init_months:
        da = index_by_exp[exp_name][init_month][freq_tag]
        leads = da.L.values

        model_lead_clim = da.mean(("Y", "M"), skipna=True).load()
        model_spread = da.std("M", skipna=True).mean("Y", skipna=True).load()
        model_init_years = infer_model_init_years(da, selected_years)
        obs_by_year_lead = obs_matrix_by_valid_times(obs_regional_for_diag, model_init_years, init_month, leads).load()
        obs_lead_clim = obs_by_year_lead.mean("Y", skipna=True).rename("obs_valid_lead_mean").load()
        bias = (model_lead_clim - obs_lead_clim).rename("bias")
        drift_rate = bias.diff("L").rename("drift_rate")
        drift_rate = drift_rate.assign_coords(L=leads[1:])
        rmse, acc = compute_rmse_and_acc(da, obs_by_year_lead)
        shock_index = shock_index_from_bias(bias, shock_leads)
        shock_index_season1 = shock_index_from_bias(bias, shock_leads_season1)

        diagnostics[exp_name][init_month] = {
            "model_lead_clim": model_lead_clim,
            "model_spread": model_spread,
            "obs_lead_clim": obs_lead_clim,
            "obs_by_year_lead": obs_by_year_lead,
            "bias": bias,
            "drift_rate": drift_rate,
            "rmse": rmse,
            "acc": acc,
            "shock_index": shock_index,
            "shock_index_season1": shock_index_season1,
        }

        print("\n", exp_name, "init", init_month)
        print("  leads:", list(leads))
        print("  calendar months:", [month_names[lead_to_cal_month(init_month, L) - 1] for L in leads])
        print("  bias:", np.round(bias.values, 4))
        print("  shock L1-L3:", round(shock_index, 4), "shock L1-L6:", round(shock_index_season1, 4))


## Plot: Forecast Evolution And Member Error For One Initialization


In [ ]:
init_year = selected_years[0]
init_month = init_months[0]
init_label = init_labels.get(init_month, f"{init_month:02d} init")

fig, axes = plt.subplots(2, 1, figsize=FIG["single_case_figsize"], sharex=True)

for exp_name in experiments:
    da = index_by_exp[exp_name][init_month][freq_tag]
    da_init = select_init_year(da, init_year).load()
    ens_mean = da_init.mean("M", skipna=True)
    ens_std = da_init.std("M", skipna=True)
    leads = ens_mean.L.values
    lead_months = leads - 1
    obs_by_lead = obs_values_by_valid_time(obs_regional_for_diag, init_year, init_month, leads)
    member_errors = da_init - obs_by_lead
    err_mean = member_errors.mean("M", skipna=True)
    err_std = member_errors.std("M", skipna=True)
    color = exp_colors[exp_name]

    axes[0].fill_between(lead_months, ens_mean - ens_std, ens_mean + ens_std, color=color, alpha=FIG["band_alpha"], linewidth=0)
    axes[0].plot(lead_months, ens_mean, marker="o", markersize=FIG["marker_size"], linewidth=FIG["line_width"], color=color, label=f"{exp_name} mean +/-1 std")
    axes[1].fill_between(lead_months, err_mean - err_std, err_mean + err_std, color=color, alpha=FIG["band_alpha"], linewidth=0)
    axes[1].plot(lead_months, err_mean, marker="o", markersize=FIG["marker_size"], linewidth=FIG["line_width"], color=color, label=f"{exp_name} error +/-1 std")

axes[0].plot(lead_months, obs_by_lead, color="black", linestyle="--", marker="s", markersize=FIG["marker_size"], linewidth=FIG["line_width"], label=reference_name)
axes[0].set_ylabel(f"{plot_name} ({plot_units})")
axes[0].set_title(f"{region_name} {plot_name} evolution | {init_label} {init_year}")
apply_axis_style(axes[0])
axes[0].legend(fontsize=FS["legend"])

axes[1].axhline(0, color="black", linewidth=FIG["thin_line_width"])
axes[1].set_xlabel("Lead time / target month")
axes[1].set_ylabel(f"Forecast error ({plot_units})")
axes[1].set_title(f"Forecast error vs {reference_name}, with ensemble error spread")
apply_axis_style(axes[1])
axes[1].legend(fontsize=FS["legend"])

xtick_labels = [f"{int(L) - 1}\n{month_names[lead_to_cal_month(init_month, L) - 1]}" for L in leads]
axes[1].set_xticks(lead_months)
axes[1].set_xticklabels(xtick_labels, fontsize=FS["tick"])
fig.tight_layout()

figname = figure_filename(field, "spinup_drift", reference_name, region_name)
figpath = FIGURE_OUTDIR / figname
mov.save_figure(
    fig, figpath,
    mode="",
    metric="drift",
    title=f"Drift Analysis: {field}",
    caption=f"{reference_name} reference, {region_name}",
    dpi=FIG["dpi"],
)
print(figpath)
plt.show()


## Plot: Lead-Time Bias, Drift Rate, And Ensemble Spread


In [ ]:
n_init = len(init_months)
fig, axes = plt.subplots(
    3,
    n_init,
    figsize=(FIG["diagnostic_col_width"] * n_init, FIG["diagnostic_fig_height"]),
    sharex="col",
    squeeze=False,
)

for col, init_month in enumerate(init_months):
    leads = index_by_exp[next(iter(experiments))][init_month][freq_tag].L.values
    lead_months = leads - 1
    init_label = init_labels.get(init_month, f"{init_month:02d} init")

    ax_bias, ax_rate, ax_spread = axes[:, col]
    for exp_name in experiments:
        color = exp_colors[exp_name]
        diag = diagnostics[exp_name][init_month]
        bias = diag["bias"]
        drift_rate = diag["drift_rate"]
        spread = diag["model_spread"]

        ax_bias.plot(lead_months, bias.values, marker="o", markersize=FIG["marker_size"], linewidth=FIG["line_width"], color=color, label=exp_name)
        ax_rate.plot(lead_months[1:], drift_rate.values, marker="o", markersize=FIG["marker_size"], linewidth=FIG["line_width"], color=color, label=exp_name)
        ax_spread.plot(lead_months, spread.values, marker="o", markersize=FIG["marker_size"], linewidth=FIG["line_width"], color=color, label=exp_name)

    for ax in (ax_bias, ax_rate, ax_spread):
        ax.axhline(0, color="black", linewidth=FIG["thin_line_width"], linestyle=":")
        ax.axvspan(spinup_span[0], spinup_span[1], color="gold", alpha=FIG["band_alpha"], label=spinup_label)
        apply_axis_style(ax)
    format_lead_time_axis(ax_bias, leads, init_month, show_labels=False)
    format_lead_time_axis(ax_rate, leads, init_month, show_labels=False)
    format_lead_time_axis(ax_spread, leads, init_month, show_labels=True)

    ax_bias.set_title(f"{init_label} - lead-time bias", loc="left")
    ax_bias.set_ylabel(f"Model - {reference_name} ({plot_units})")
    ax_rate.set_title("Drift rate", loc="left")
    ax_rate.set_ylabel(f"Delta bias / lead ({plot_units})")
    ax_spread.set_title("Ensemble spread", loc="left")
    ax_spread.set_ylabel(f"Member std ({plot_units})")
    ax_spread.set_xlabel("Lead time / target month")
    ax_bias.legend(fontsize=FS["legend"])

fig.suptitle(f"{region_name} {plot_name} lead-time drift diagnostics", fontsize=FS["suptitle"], y=1.01)
fig.tight_layout()

figname = figure_filename(field, "drift_bias_spread", reference_name, region_name)
figpath = FIGURE_OUTDIR / figname
mov.save_figure(
    fig, figpath,
    mode="",
    metric="drift",
    title=f"Drift Analysis: {field}",
    caption=f"{reference_name} reference, {region_name}",
    dpi=FIG["dpi"],
)
print(figpath)
plt.show()


## Plot: Bias By Verification Calendar Month


In [ ]:
from matplotlib.lines import Line2D

n_init = len(init_months)
fig, axes = plt.subplots(
    1,
    n_init,
    figsize=(FIG["season_col_width"] * n_init, FIG["season_fig_height"]),
    sharey=True,
    squeeze=False,
)
axes = axes[0]
month_cmap = plt.get_cmap("tab20", 12)
month_colors = {month: month_cmap(month - 1) for month in range(1, 13)}
month_handles = [
    Line2D([0], [0], marker="o", linestyle="", color=month_colors[month], label=month_names[month - 1])
    for month in range(1, 13)
]

for ax, init_month in zip(axes, init_months):
    exp_handles = []
    for exp_name in experiments:
        diag = diagnostics[exp_name][init_month]
        bias = diag["bias"]
        leads = bias.L.values
        lead_months = leads - 1
        cal_months = np.array([lead_to_cal_month(init_month, L) for L in leads])
        color = exp_colors[exp_name]

        line = ax.plot(
            lead_months,
            bias.values,
            color=color,
            linewidth=FIG["line_width"] * 0.9,
            alpha=FIG["line_alpha"],
            label=exp_name,
        )[0]
        exp_handles.append(line)

        for month in range(1, 13):
            mask = cal_months == month
            if not np.any(mask):
                continue
            ax.scatter(
                lead_months[mask],
                bias.values[mask],
                s=FIG["scatter_size"],
                color=month_colors[month],
                edgecolors=color,
                linewidths=FIG["scatter_edge_width"],
                zorder=3,
            )

    ax.axhline(0, color="black", linewidth=FIG["thin_line_width"], linestyle=":")
    ax.axvspan(spinup_span[0], spinup_span[1], color="gold", alpha=FIG["band_alpha"])
    ax.set_title(f"{init_labels.get(init_month, init_month)} - bias with verification month markers", loc="left")
    ax.set_xlabel("Lead time (months)")
    ax.set_ylabel(f"Bias ({plot_units})")
    apply_axis_style(ax)
    ax.legend(handles=exp_handles, fontsize=FS["legend"], title_fontsize=FS["legend_title"], loc="best", title="Experiment")

fig.legend(handles=month_handles, fontsize=FS["legend"], title_fontsize=FS["legend_title"], ncol=6, loc="lower center", title="Verification month")
fig.suptitle(f"{region_name} {plot_name} season-dependent drift", fontsize=FS["suptitle"], y=1.02)
fig.tight_layout(rect=(0, 0.13, 1, 1))

figname = figure_filename(field, "drift_calendar_bias", reference_name, region_name)
figpath = FIGURE_OUTDIR / figname
mov.save_figure(
    fig, figpath,
    mode="",
    metric="drift",
    title=f"Drift Analysis: {field}",
    caption=f"{reference_name} reference, {region_name}",
    dpi=FIG["dpi"],
)
print(figpath)
plt.show()


## Surface Temperature Companion Plots


In [ ]:
from matplotlib.lines import Line2D

active_surface_plot_fields = [plot_field for plot_field in surface_plot_fields if plot_field in index_by_field and plot_field != field]
comparison_precip_field = comparison_fields.get("precip", "PRECT")
comparison_sst_field = comparison_fields.get("sst", "SST")


def load_obs_regional_for_field(field_key):
    cfg = VARIABLE_CONFIG[field_key]
    if cfg.get("obs_product") is None:
        return None

    native = cfg.get("native_field", MODEL_FIELD_ALIASES.get(field_key, field_key))
    obs_name = cfg["obs_var"]
    obs_chunk = {"time": 120, "lat": 90, "lon": 180}
    lat_name = "lat"
    lon_name = "lon"

    ds = obs_access.get_monthly_data(
        obs_dir=obs_dir,
        field=native,
        field_map={native: obs_name},
        product=cfg["obs_product"],
        start_year=cfg["obs_yrs"],
        end_year=cfg["obs_yre"],
        chunks=obs_chunk,
        verbose=True,
    )
    if obs_name not in ds.data_vars:
        raise KeyError(f"Observation variable {obs_name!r} not found for {field_key}. Available: {list(ds.data_vars)}")

    obs_da = cfg.get("obs_convert", no_conversion)(ds[obs_name])
    ds[obs_name] = obs_da
    ds = ds.sel(time=slice(str(min(selected_years)), str(min(cfg["obs_yre"], max(selected_years) + 3))))

    landmask = spatial.create_land_sea_mask(ds[obs_name], lat_key=lat_name, lon_key=lon_name).astype(bool)
    oceanmask = (~landmask).chunk({lat_name: 90, lon_name: 180})
    weights = obs_access.obs_regional_weights(
        ds[obs_name],
        region,
        lat_name=lat_name,
        lon_name=lon_name,
        mask=oceanmask,
    )
    obs_regional = ds[obs_name].weighted(weights).mean((lat_name, lon_name)).load()
    obs_regional.name = f"{field_key}_obs_regional_index"
    obs_regional.attrs["units"] = cfg["plot_units"]

    if freq_tag == "mon":
        return obs_regional
    if freq_tag == "seas":
        return (
            obs_regional.rolling(time=3, min_periods=3, center=True)
            .mean()
            .dropna("time", how="all")
            .rename(f"{field_key}_obs_regional_index_seasonal")
        )
    raise ValueError(f"Unsupported freq_tag={freq_tag!r}; expected 'mon' or 'seas'.")


def build_field_diagnostics(field_key, obs_series):
    out = {}
    for exp_name in experiments:
        out[exp_name] = {}
        for init_month in init_months:
            da = index_by_field[field_key][exp_name][init_month][freq_tag]
            leads = da.L.values
            model_lead_clim = da.mean(("Y", "M"), skipna=True).load()
            model_spread = da.std("M", skipna=True).mean("Y", skipna=True).load()
            model_init_years = infer_model_init_years(da, selected_years)
            obs_by_year_lead = obs_matrix_by_valid_times(obs_series, model_init_years, init_month, leads).load()
            obs_lead_clim = obs_by_year_lead.mean("Y", skipna=True).rename("obs_valid_lead_mean").load()
            bias = (model_lead_clim - obs_lead_clim).rename("bias")
            drift_rate = bias.diff("L").rename("drift_rate").assign_coords(L=leads[1:])
            rmse, acc = compute_rmse_and_acc(da, obs_by_year_lead)
            out[exp_name][init_month] = {
                "model_lead_clim": model_lead_clim,
                "model_spread": model_spread,
                "obs_lead_clim": obs_lead_clim,
                "obs_by_year_lead": obs_by_year_lead,
                "bias": bias,
                "drift_rate": drift_rate,
                "rmse": rmse,
                "acc": acc,
                "shock_index": shock_index_from_bias(bias, shock_leads),
                "shock_index_season1": shock_index_from_bias(bias, shock_leads_season1),
            }
    return out


def plot_bias_rate_spread_for_field(field_key, field_diagnostics):
    cfg = VARIABLE_CONFIG[field_key]
    units = cfg["plot_units"]
    name = cfg["plot_name"]
    n_init = len(init_months)
    fig, axes = plt.subplots(
        3,
        n_init,
        figsize=(FIG["diagnostic_col_width"] * n_init, FIG["diagnostic_fig_height"]),
        sharex="col",
        squeeze=False,
    )

    for col, init_month in enumerate(init_months):
        leads = index_by_field[field_key][next(iter(experiments))][init_month][freq_tag].L.values
        lead_months = leads - 1
        init_label = init_labels.get(init_month, f"{init_month:02d} init")
        ax_bias, ax_rate, ax_spread = axes[:, col]

        for exp_name in experiments:
            color = exp_colors[exp_name]
            diag = field_diagnostics[exp_name][init_month]
            ax_bias.plot(lead_months, diag["bias"].values, marker="o", markersize=FIG["marker_size"], linewidth=FIG["line_width"], color=color, label=exp_name)
            ax_rate.plot(lead_months[1:], diag["drift_rate"].values, marker="o", markersize=FIG["marker_size"], linewidth=FIG["line_width"], color=color, label=exp_name)
            ax_spread.plot(lead_months, diag["model_spread"].values, marker="o", markersize=FIG["marker_size"], linewidth=FIG["line_width"], color=color, label=exp_name)

        for ax in (ax_bias, ax_rate, ax_spread):
            ax.axhline(0, color="black", linewidth=FIG["thin_line_width"], linestyle=":")
            ax.axvspan(spinup_span[0], spinup_span[1], color="gold", alpha=FIG["band_alpha"], label=spinup_label)
            apply_axis_style(ax)
        format_lead_time_axis(ax_bias, leads, init_month, show_labels=False)
        format_lead_time_axis(ax_rate, leads, init_month, show_labels=False)
        format_lead_time_axis(ax_spread, leads, init_month, show_labels=True)

        ax_bias.set_title(f"{init_label} - lead-time bias", loc="left")
        ax_bias.set_ylabel(f"Model - {reference_name} ({units})")
        ax_rate.set_title("Drift rate", loc="left")
        ax_rate.set_ylabel(f"Delta bias / lead ({units})")
        ax_spread.set_title("Ensemble spread", loc="left")
        ax_spread.set_ylabel(f"Member std ({units})")
        ax_spread.set_xlabel("Lead time / target month")
        ax_bias.legend(fontsize=FS["legend"])

    fig.suptitle(f"{region_name} {name} lead-time drift diagnostics", fontsize=FS["suptitle"], y=1.01)
    fig.tight_layout()
    figpath = FIGURE_OUTDIR / figure_filename(field_key, "drift_bias_spread", reference_name, region_name)
    mov.save_figure(
        fig, figpath,
        mode="",
        metric="drift",
        title=f"Drift Analysis: {field}",
        caption=f"{reference_name} reference, {region_name}",
        dpi=FIG["dpi"],
    )
    print(figpath)
    plt.show()


def plot_calendar_month_bias_for_field(field_key, field_diagnostics):
    cfg = VARIABLE_CONFIG[field_key]
    units = cfg["plot_units"]
    name = cfg["plot_name"]
    n_init = len(init_months)
    fig, axes = plt.subplots(
        1,
        n_init,
        figsize=(FIG["season_col_width"] * n_init, FIG["season_fig_height"]),
        sharey=True,
        squeeze=False,
    )
    axes = axes[0]
    month_cmap = plt.get_cmap("tab20", 12)
    month_colors = {month: month_cmap(month - 1) for month in range(1, 13)}
    month_handles = [
        Line2D([0], [0], marker="o", linestyle="", color=month_colors[month], label=month_names[month - 1])
        for month in range(1, 13)
    ]

    for ax, init_month in zip(axes, init_months):
        exp_handles = []
        for exp_name in experiments:
            bias = field_diagnostics[exp_name][init_month]["bias"]
            leads = bias.L.values
            lead_months = leads - 1
            cal_months = np.array([lead_to_cal_month(init_month, L) for L in leads])
            color = exp_colors[exp_name]
            line = ax.plot(
                lead_months,
                bias.values,
                color=color,
                linewidth=FIG["line_width"] * 0.9,
                alpha=FIG["line_alpha"],
                label=exp_name,
            )[0]
            exp_handles.append(line)
            for month in range(1, 13):
                mask = cal_months == month
                if not np.any(mask):
                    continue
                ax.scatter(
                    lead_months[mask],
                    bias.values[mask],
                    s=FIG["scatter_size"],
                    color=month_colors[month],
                    edgecolors=color,
                    linewidths=FIG["scatter_edge_width"],
                    zorder=3,
                )
        ax.axhline(0, color="black", linewidth=FIG["thin_line_width"], linestyle=":")
        ax.axvspan(spinup_span[0], spinup_span[1], color="gold", alpha=FIG["band_alpha"])
        ax.set_title(f"{init_labels.get(init_month, init_month)} - bias with verification month markers", loc="left")
        ax.set_xlabel("Lead time (months)")
        ax.set_ylabel(f"Bias ({units})")
        apply_axis_style(ax)
        ax.legend(handles=exp_handles, fontsize=FS["legend"], title_fontsize=FS["legend_title"], loc="best", title="Experiment")

    fig.legend(handles=month_handles, fontsize=FS["legend"], title_fontsize=FS["legend_title"], ncol=6, loc="lower center", title="Verification month")
    fig.suptitle(f"{region_name} {name} season-dependent drift", fontsize=FS["suptitle"], y=1.02)
    fig.tight_layout(rect=(0, 0.13, 1, 1))
    figpath = FIGURE_OUTDIR / figure_filename(field_key, "drift_calendar_bias", reference_name, region_name)
    mov.save_figure(
        fig, figpath,
        mode="",
        metric="drift",
        title=f"Drift Analysis: {field}",
        caption=f"{reference_name} reference, {region_name}",
        dpi=FIG["dpi"],
    )
    print(figpath)
    plt.show()


surface_diagnostics_by_field = {}
for plot_field in active_surface_plot_fields:
    obs_series = load_obs_regional_for_field(plot_field)
    if obs_series is None:
        print(f"Skipping {plot_field}: no observation product configured.")
        continue
    surface_diagnostics_by_field[plot_field] = build_field_diagnostics(plot_field, obs_series)
    plot_bias_rate_spread_for_field(plot_field, surface_diagnostics_by_field[plot_field])
    plot_calendar_month_bias_for_field(plot_field, surface_diagnostics_by_field[plot_field])


def plot_precip_sst_comparison(sst_diagnostics, precip_field=None, sst_field=None):
    precip_field = precip_field or comparison_precip_field
    sst_field = sst_field or comparison_sst_field
    if field != precip_field:
        print(f"Skipping {precip_field}-{sst_field} comparison because primary field is {field!r}, not {precip_field!r}.")
        return
    precip_cfg = VARIABLE_CONFIG[precip_field]
    sst_cfg = VARIABLE_CONFIG[sst_field]
    precip_name = precip_cfg["plot_name"].upper() if precip_field == "PRECT" else precip_cfg["plot_name"]
    sst_name = sst_cfg["plot_name"]
    precip_units = precip_cfg["plot_units"]
    sst_units = sst_cfg["plot_units"]

    n_init = len(init_months)
    fig, axes = plt.subplots(
        3,
        n_init,
        figsize=(7.4 * n_init, 11.2),
        squeeze=False,
        constrained_layout=True,
    )
    marker_cycle = ["o", "s", "^", "D", "P", "X"]
    marker_by_exp = {
        exp_name: experiment_markers.get(exp_name, marker_cycle[i % len(marker_cycle)])
        for i, exp_name in enumerate(experiments)
    }

    def lead_axis_ticks(leads, init_month):
        lead_months = np.asarray(leads) - 1
        tick_idx = list(range(0, len(lead_months), 3))
        if len(lead_months) and tick_idx[-1] != len(lead_months) - 1:
            tick_idx.append(len(lead_months) - 1)
        major_ticks = lead_months[tick_idx]
        major_labels = [
            f"L{int(lead_months[i])}\n{month_names[lead_to_cal_month(init_month, leads[i]) - 1]}"
            for i in tick_idx
        ]
        return lead_months, major_ticks, major_labels

    for col, init_month in enumerate(init_months):
        init_label = init_labels.get(init_month, f"{init_month:02d} init")
        precip_ax, sst_ax, scatter_ax = axes[:, col]
        lead_months_ref = None
        major_ticks_ref = None
        major_labels_ref = None
        all_sst_bias = []
        all_prect_bias = []

        for exp_name in experiments:
            color = exp_colors[exp_name]
            precip_bias = diagnostics[exp_name][init_month]["bias"]
            sst_bias = sst_diagnostics[exp_name][init_month]["bias"]
            common_leads = np.intersect1d(precip_bias.L.values, sst_bias.L.values)
            precip_bias = precip_bias.sel(L=common_leads)
            sst_bias = sst_bias.sel(L=common_leads)
            lead_months, major_ticks, major_labels = lead_axis_ticks(common_leads, init_month)
            lead_months_ref = lead_months
            major_ticks_ref = major_ticks
            major_labels_ref = major_labels
            all_sst_bias.extend(np.asarray(sst_bias.values, dtype=float))
            all_prect_bias.extend(np.asarray(precip_bias.values, dtype=float))

            precip_ax.plot(
                lead_months,
                precip_bias.values,
                marker="o",
                linewidth=FIG["line_width"],
                markersize=FIG["marker_size"],
                color=color,
                label=exp_name,
            )
            sst_ax.plot(
                lead_months,
                sst_bias.values,
                marker="o",
                linewidth=FIG["line_width"],
                markersize=FIG["marker_size"],
                color=color,
                label=exp_name,
            )

            scatter_ax.scatter(
                sst_bias.values,
                precip_bias.values,
                s=FIG["scatter_size"] * 1.45,
                marker=marker_by_exp[exp_name],
                facecolors=color,
                edgecolors="white",
                linewidths=0.8,
                label=exp_name,
                alpha=0.9,
                zorder=3,
            )
            for lead, xval, yval in zip(common_leads, sst_bias.values, precip_bias.values):
                if int(lead) not in comparison_scatter_label_leads:
                    continue
                scatter_ax.annotate(
                    f"L{int(lead) - 1}",
                    (xval, yval),
                    xytext=(4, 4),
                    textcoords="offset points",
                    fontsize=max(8, FS["tick"] - 2),
                    color=color,
                    alpha=0.9,
                )

        for ax in (precip_ax, sst_ax):
            ax.axhline(0, color="black", linewidth=FIG["thin_line_width"], linestyle=":")
            ax.axvspan(spinup_span[0], spinup_span[1], color="gold", alpha=FIG["band_alpha"])
            apply_axis_style(ax)
            if lead_months_ref is not None:
                ax.set_xlim(float(np.nanmin(lead_months_ref)) - 0.75, float(np.nanmax(lead_months_ref)) + 0.75)
                ax.set_xticks(major_ticks_ref)
                ax.set_xticks(lead_months_ref, minor=True)
                ax.grid(which="minor", axis="x", alpha=0.16, linewidth=FIG["thin_line_width"])
        precip_ax.tick_params(axis="x", labelbottom=False)
        sst_ax.set_xticklabels(major_labels_ref, fontsize=FS["tick"])

        scatter_ax.axhline(0, color="black", linewidth=FIG["thin_line_width"], linestyle=":")
        scatter_ax.axvline(0, color="black", linewidth=FIG["thin_line_width"], linestyle=":")
        finite_sst = np.asarray([v for v in all_sst_bias if np.isfinite(v)])
        finite_prect = np.asarray([v for v in all_prect_bias if np.isfinite(v)])
        if finite_sst.size:
            pad = max(0.2, 0.08 * (finite_sst.max() - finite_sst.min()))
            scatter_ax.set_xlim(finite_sst.min() - pad, finite_sst.max() + pad)
        if finite_prect.size:
            pad = max(0.2, 0.08 * (finite_prect.max() - finite_prect.min()))
            scatter_ax.set_ylim(finite_prect.min() - pad, finite_prect.max() + pad)
        apply_axis_style(scatter_ax)
        scatter_ax.set_xlabel(f"{sst_name} bias ({sst_units})")
        scatter_ax.set_ylabel(f"{precip_name} bias ({precip_units})")
        scatter_ax.set_title(f"{init_label} - linked bias by lead", loc="left")
        scatter_ax.legend(fontsize=FS["legend"], loc="best", frameon=True)

        precip_ax.set_title(f"{init_label} - {precip_name} bias", loc="left")
        precip_ax.set_ylabel(f"{precip_name} bias ({precip_units})")
        sst_ax.set_title(f"{init_label} - {sst_name} bias", loc="left")
        sst_ax.set_ylabel(f"{sst_name} bias ({sst_units})")
        sst_ax.set_xlabel("Lead time and target month")
        precip_ax.legend(fontsize=FS["legend"], loc="best")

    fig.suptitle(f"{region_name} {precip_name}-{sst_name} lead-time bias comparison", fontsize=FS["suptitle"])
    figpath = FIGURE_OUTDIR / figure_filename(precip_field, sst_field, "drift_bias_comparison", reference_name, region_name)
    mov.save_figure(
        fig, figpath,
        mode="",
        metric="drift",
        title=f"Drift Analysis: {field}",
        caption=f"{reference_name} reference, {region_name}",
        dpi=FIG["dpi"],
    )
    print(figpath)
    plt.show()


if make_precip_sst_comparison and comparison_sst_field in surface_diagnostics_by_field:
    plot_precip_sst_comparison(surface_diagnostics_by_field[comparison_sst_field], comparison_precip_field, comparison_sst_field)

if not surface_diagnostics_by_field:
    print("No extra surface-temperature companion plots were made. Check surface_plot_fields, fields, and available observations.")


## Coupled Consistency Diagnostics


In [ ]:
consistency_diagnostics = {}

required_for_sst_t2m = {"SST", "TREFHT"}
required_for_heat_flux = {"FSNS", "FLNS", "LHFLX", "SHFLX"}
loaded_fields = set(index_by_field)

for exp_name in experiments:
    consistency_diagnostics[exp_name] = {}
    for init_month in init_months:
        derived = {}

        if required_for_sst_t2m <= loaded_fields:
            derived["SST_minus_T2m"] = (
                index_by_field["SST"][exp_name][init_month][freq_tag]
                - index_by_field["TREFHT"][exp_name][init_month][freq_tag]
            ).rename("SST_minus_T2m")
            derived["SST_minus_T2m"].attrs["units"] = "degC"

        if required_for_heat_flux <= loaded_fields:
            derived["net_surface_heat_flux"] = (
                index_by_field["FSNS"][exp_name][init_month][freq_tag]
                - index_by_field["FLNS"][exp_name][init_month][freq_tag]
                - index_by_field["LHFLX"][exp_name][init_month][freq_tag]
                - index_by_field["SHFLX"][exp_name][init_month][freq_tag]
            ).rename("net_surface_heat_flux")
            derived["net_surface_heat_flux"].attrs["units"] = "W/m2"

        consistency_diagnostics[exp_name][init_month] = {
            name: da.mean(("Y", "M"), skipna=True).load()
            for name, da in derived.items()
        }

consistency_rows = []
for exp_name in experiments:
    for init_month in init_months:
        for diagnostic_name, da in consistency_diagnostics[exp_name][init_month].items():
            consistency_rows.append(
                {
                    "experiment": exp_name,
                    "init_month": init_month,
                    "diagnostic": diagnostic_name,
                    "early_mean": safe_sel_mean(da, shock_leads_season1),
                    "month_1": float(da.sel(L=1)),
                    "lead_6": float(da.sel(L=6)) if 6 in da.L else np.nan,
                    "units": da.attrs.get("units", ""),
                }
            )

consistency_summary_df = pd.DataFrame(consistency_rows)
if len(consistency_summary_df):
    display(consistency_summary_df)
else:
    print("No coupled consistency diagnostics were computed; check consistency_fields and available cached fields.")


if make_consistency_field_figures:
    active_consistency_fields = [field_key for field_key in consistency_fields if field_key in index_by_field]
    if active_consistency_fields:
        for init_month in init_months:
            fig, axes = plt.subplots(
                1,
                len(experiments),
                figsize=(7.2 * len(experiments), max(5.2, 0.42 * len(active_consistency_fields) + 2.4)),
                sharex=True,
                sharey=True,
                squeeze=False,
                constrained_layout=True,
            )
            axes = axes[0]
            image = None
            for ax, exp_name in zip(axes, experiments):
                rows = []
                row_labels = []
                leads_ref = None
                for field_key in active_consistency_fields:
                    da = index_by_field[field_key][exp_name][init_month][freq_tag]
                    lead_clim = da.mean(("Y", "M"), skipna=True).load()
                    leads = lead_clim.L.values
                    leads_ref = leads
                    lead_adjustment = lead_clim - lead_clim.sel(L=leads[0])
                    scale = float(lead_adjustment.std("L", skipna=True))
                    if np.isfinite(scale) and scale > 0:
                        lead_adjustment = lead_adjustment / scale
                    rows.append(lead_adjustment.values)
                    row_labels.append(field_key)

                matrix = np.asarray(rows, dtype=float)
                vmax = np.nanpercentile(np.abs(matrix), 95) if np.isfinite(matrix).any() else 1.0
                if not np.isfinite(vmax) or vmax == 0:
                    vmax = 1.0
                image = ax.imshow(matrix, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
                ax.set_title(f"{init_labels.get(init_month, init_month)} | {exp_name}", loc="left")
                ax.set_yticks(np.arange(len(row_labels)))
                ax.set_yticklabels(row_labels)
                ax.set_ylabel("Consistency field")
                if leads_ref is not None:
                    tick_idx = list(range(0, len(leads_ref), lead_label_every))
                    if tick_idx and tick_idx[-1] != len(leads_ref) - 1:
                        tick_idx.append(len(leads_ref) - 1)
                    ax.set_xticks(tick_idx)
                    ax.set_xticklabels([f"L{int(leads_ref[i]) - 1}" for i in tick_idx], fontsize=FS["tick"])
                    ax.set_xticks(np.arange(-0.5, len(leads_ref), 1), minor=True)
                ax.set_yticks(np.arange(-0.5, len(row_labels), 1), minor=True)
                ax.grid(which="minor", color="white", linewidth=FIG["thin_line_width"])
                ax.tick_params(which="minor", bottom=False, left=False)
                ax.set_xlabel("Lead time")
                ax.tick_params(labelsize=FS["tick"])

            if image is not None:
                cbar = fig.colorbar(image, ax=axes, orientation="vertical", fraction=0.025, pad=0.02)
                cbar.set_label("Normalized adjustment from L0", fontsize=FS["colorbar"])
                cbar.ax.tick_params(labelsize=FS["tick"])
            fig.suptitle(f"{region_name}: consistency-field lead-time adjustment", fontsize=FS["suptitle"])
            figpath = FIGURE_OUTDIR / figure_filename("consistency_adjustment_heatmap", region_name)
            mov.save_figure(
                fig, figpath,
                mode="",
                metric="drift",
                title=f"Drift Analysis: {field}",
                caption=f"{reference_name} reference, {region_name}",
                dpi=FIG["dpi"],
            )
            print(figpath)
            plt.show()
    else:
        print("No consistency field figures were made; none of consistency_fields are loaded.")

if make_consistency_field_figures and len(consistency_summary_df):
    derived_names = sorted(consistency_summary_df["diagnostic"].unique())
    fig, axes = plt.subplots(
        len(derived_names),
        len(init_months),
        figsize=(7.0 * len(init_months), 3.8 * len(derived_names)),
        sharex=True,
        squeeze=False,
        constrained_layout=True,
    )
    for row, diagnostic_name in enumerate(derived_names):
        for col, init_month in enumerate(init_months):
            ax = axes[row, col]
            leads_for_axis = None
            for exp_name in experiments:
                da = consistency_diagnostics[exp_name][init_month].get(diagnostic_name)
                if da is None:
                    continue
                leads_for_axis = da.L.values
                ax.plot(da.L - 1, da.values, marker="o", linewidth=FIG["line_width"], markersize=FIG["marker_size"], color=exp_colors.get(exp_name), label=exp_name)
            ax.axhline(0, color="0.25", linewidth=FIG["thin_line_width"], linestyle=":")
            ax.set_title(f"{init_labels.get(init_month, init_month)} | {diagnostic_name}", loc="left")
            units = consistency_summary_df.loc[consistency_summary_df["diagnostic"].eq(diagnostic_name), "units"].iloc[0]
            ax.set_ylabel(units)
            if leads_for_axis is not None:
                format_lead_time_axis(ax, leads_for_axis, init_month, show_labels=(row == len(derived_names) - 1))
            if row == len(derived_names) - 1:
                ax.set_xlabel("Lead time and target month")
            apply_axis_style(ax)
            if row == 0 and col == len(init_months) - 1:
                ax.legend(frameon=False, fontsize=FS["legend"])
    fig.suptitle(f"{region_name}: derived coupled consistency diagnostics", fontsize=FS["suptitle"])
    figpath = FIGURE_OUTDIR / figure_filename("consistency_derived_lines", region_name)
    mov.save_figure(
        fig, figpath,
        mode="",
        metric="drift",
        title=f"Drift Analysis: {field}",
        caption=f"{reference_name} reference, {region_name}",
        dpi=FIG["dpi"],
    )
    print(figpath)
    plt.show()


## Numeric Summary Table


In [ ]:
rows = []
for exp_name in experiments:
    for init_month in init_months:
        diag = diagnostics[exp_name][init_month]
        bias = diag["bias"]
        spread = diag["model_spread"]
        model_clim = diag["model_lead_clim"]
        obs_valid_mean = diag["obs_lead_clim"]
        rmse = diag["rmse"]
        acc = diag["acc"]
        for L in bias.L.values:
            rows.append(
                {
                    "experiment": exp_name,
                    "init_month": int(init_month),
                    "lead": int(L),
                    "lead_month": int(L) - 1,
                    "calendar_month": month_names[lead_to_cal_month(init_month, L) - 1],
                    "model_clim": float(model_clim.sel(L=L)),
                    "obs_valid_mean": float(obs_valid_mean.sel(L=L)),
                    "bias": float(bias.sel(L=L)),
                    "spread": float(spread.sel(L=L)),
                    "rmse": float(rmse.sel(L=L)),
                    "acc": float(acc.sel(L=L)),
                }
            )

summary = xr.Dataset(
    {
        "model_clim": ("row", [r["model_clim"] for r in rows]),
        "obs_valid_mean": ("row", [r["obs_valid_mean"] for r in rows]),
        "bias": ("row", [r["bias"] for r in rows]),
        "spread": ("row", [r["spread"] for r in rows]),
        "rmse": ("row", [r["rmse"] for r in rows]),
        "acc": ("row", [r["acc"] for r in rows]),
    },
    coords={
        "experiment": ("row", [r["experiment"] for r in rows]),
        "init_month": ("row", [r["init_month"] for r in rows]),
        "lead": ("row", [r["lead"] for r in rows]),
        "lead_month": ("row", [r["lead_month"] for r in rows]),
        "calendar_month": ("row", [r["calendar_month"] for r in rows]),
    },
)
summary.attrs.update(field=field, region=region_name, units=plot_units)
summary_df = pd.DataFrame(rows)
summary_df["init_label"] = summary_df["init_month"].map(lambda m: init_labels.get(m, f"{m:02d} init"))
summary_df["case"] = summary_df["init_label"] + " | " + summary_df["experiment"]

case_summary_rows = []
for exp_name in experiments:
    for init_month in init_months:
        diag = diagnostics[exp_name][init_month]
        bias = diag["bias"]
        case_summary_rows.append(
            {
                "experiment": exp_name,
                "init_month": int(init_month),
                "init_label": init_labels.get(init_month, f"{init_month:02d} init"),
                "early_shock": diag["shock_index"],
                "early_shock_season1": diag["shock_index_season1"],
                "month_1_bias": float(bias.sel(L=1)),
                "season_1_drift": season1_drift_from_bias(bias, shock_leads_season1),
                "spread": safe_sel_mean(diag["model_spread"], skill_leads),
                "RMSE": safe_sel_mean(diag["rmse"], skill_leads),
                "ACC": safe_sel_mean(diag["acc"], skill_leads),
            }
        )
case_summary_df = pd.DataFrame(case_summary_rows)
if show_case_summary_table:
    display(
        case_summary_df.style.format(
            {
                "early_shock": "{:.3f}",
                "early_shock_season1": "{:.3f}",
                "month_1_bias": "{:+.3f}",
                "season_1_drift": "{:+.3f}",
                "spread": "{:.3f}",
                "RMSE": "{:.3f}",
                "ACC": "{:.3f}",
            }
        ).set_caption(f"{region_name} {plot_name} compact initialization summary ({plot_units})")
    )

bias_abs = max(abs(summary_df["bias"].min()), abs(summary_df["bias"].max()))
summary_compare = summary_df.pivot_table(
    index=["init_month", "init_label", "lead", "lead_month", "calendar_month", "obs_valid_mean"],
    columns="experiment",
    values=["model_clim", "bias", "spread", "rmse", "acc"],
    aggfunc="first",
    dropna=False,
).reset_index()
summary_compare.columns = [
    "_".join(str(part) for part in col if part != "") if isinstance(col, tuple) else str(col)
    for col in summary_compare.columns
]
summary_compare = summary_compare.sort_values(["init_month", "lead"]).drop(columns="init_month")

exp_names = list(experiments)
if len(exp_names) >= 2:
    ref_exp, test_exp = exp_names[0], exp_names[1]
    for metric in ["bias", "rmse", "acc"]:
        ref_col = f"{metric}_{ref_exp}"
        test_col = f"{metric}_{test_exp}"
        if ref_col in summary_compare.columns and test_col in summary_compare.columns:
            summary_compare[f"{metric}_diff_{test_exp}_minus_{ref_exp}"] = summary_compare[test_col] - summary_compare[ref_col]

base_cols = ["init_label", "lead", "lead_month", "calendar_month", "obs_valid_mean"]
metric_cols = [
    f"{metric}_{exp_name}"
    for metric in ["model_clim", "spread", "bias", "rmse", "acc"]
    for exp_name in exp_names
    if f"{metric}_{exp_name}" in summary_compare.columns
]
diff_cols = [c for c in summary_compare.columns if c.startswith(("bias_diff_", "rmse_diff_", "acc_diff_"))]
summary_compare = summary_compare[[c for c in base_cols + metric_cols + diff_cols if c in summary_compare.columns]]

format_map = {col: "{:.3f}" for col in summary_compare.columns if col.startswith(("model_clim_", "obs_valid_mean", "spread_", "rmse_"))}
format_map.update({col: "{:+.3f}" for col in summary_compare.columns if col.startswith(("bias_", "bias_diff_", "rmse_diff_", "acc_diff_"))})
format_map.update({col: "{:.3f}" for col in summary_compare.columns if col.startswith("acc_") and not col.startswith("acc_diff_")})
bias_cols = [col for col in summary_compare.columns if col.startswith("bias_")]
spread_cols = [col for col in summary_compare.columns if col.startswith("spread_")]
rmse_cols = [col for col in summary_compare.columns if col.startswith("rmse_")]
acc_cols = [col for col in summary_compare.columns if col.startswith("acc_")]

styled_summary = (
    summary_compare.style.format(format_map)
    .background_gradient(subset=bias_cols, cmap="RdBu_r", vmin=-bias_abs, vmax=bias_abs)
    .background_gradient(subset=spread_cols, cmap="YlOrRd")
    .background_gradient(subset=rmse_cols, cmap="YlOrRd")
    .background_gradient(subset=acc_cols, cmap="YlGn")
    .set_table_styles(
        [
            {"selector": "caption", "props": [("font-size", f"{FS['label']}pt"), ("font-weight", "bold")]},
            {"selector": "th", "props": [("font-size", f"{FS['table']}pt")]},
            {"selector": "td", "props": [("font-size", f"{FS['table']}pt")]},
        ]
    )
    .set_caption(f"{region_name} {plot_name} lead-time drift and skill comparison, same valid years/months ({plot_units})")
)
if show_lead_detail_table:
    display(styled_summary)
else:
    print("Lead-by-lead numeric table hidden. Set show_lead_detail_table=True in the User Control Panel to display it.")

if write_summary_text_files:
    summary_table_outdir.mkdir(parents=True, exist_ok=True)
    year_label = f"{min(selected_years)}_{max(selected_years)}"
    summary_prefix = "_".join(
        safe_token(part).lower()
        for part in [field, region_name, freq_tag, reference_name, year_label]
        if str(part).strip()
    )
    table_exports = {
        "compact_summary": case_summary_df,
        "lead_detail": summary_df,
        "experiment_comparison": summary_compare,
    }
    written_paths = []
    for table_name, table_df in table_exports.items():
        csv_path = summary_table_outdir / f"{summary_prefix}_{table_name}.csv"
        txt_path = summary_table_outdir / f"{summary_prefix}_{table_name}.txt"
        table_df.to_csv(csv_path, index=False, float_format=summary_float_format)
        txt_path.write_text(table_df.to_string(index=False, float_format=lambda value: summary_float_format % value))
        written_paths.extend([csv_path, txt_path])

    readme_path = summary_table_outdir / f"{summary_prefix}_README.txt"
    readme_path.write_text(
        "S2D initialization consistency/drift summary tables\n"
        f"field: {field}\n"
        f"region: {region_name} {region}\n"
        f"experiments: {list(experiments)}\n"
        f"init_months: {init_months}\n"
        f"selected_years setting: {selected_years}\n"
        f"frequency: {freq_tag}\n\n"
        "Files:\n"
        "- compact_summary: one row per experiment/init month; best quick audit table.\n"
        "- lead_detail: one row per experiment/init month/lead.\n"
        "- experiment_comparison: lead-by-lead pivot table comparing experiments.\n"
    )
    written_paths.append(readme_path)
    print("Wrote numeric summary backups:")
    for path in written_paths:
        print(" ", path)

lead_order = sorted(summary_df["lead"].unique())
heatmap_tick_idx = list(range(0, len(lead_order), lead_label_every))
if heatmap_tick_idx and heatmap_tick_idx[-1] != len(lead_order) - 1:
    heatmap_tick_idx.append(len(lead_order) - 1)
heatmap_tick_positions = np.asarray(heatmap_tick_idx)
heatmap_tick_labels = [f"L{int(lead_order[i]) - 1}" for i in heatmap_tick_idx]
case_order = [
    f"{init_labels.get(init_month, f'{init_month:02d} init')} | {exp_name}"
    for init_month in init_months
    for exp_name in experiments
]

bias_grid = summary_df.pivot(index="case", columns="lead", values="bias").reindex(index=case_order, columns=lead_order)
spread_grid = summary_df.pivot(index="case", columns="lead", values="spread").reindex(index=case_order, columns=lead_order)
rmse_grid = summary_df.pivot(index="case", columns="lead", values="rmse").reindex(index=case_order, columns=lead_order)
acc_grid = summary_df.pivot(index="case", columns="lead", values="acc").reindex(index=case_order, columns=lead_order)

fig, axes = plt.subplots(
    4,
    1,
    figsize=(
        max(FIG["summary_min_width"], FIG["summary_lead_width"] * len(lead_order)),
        FIG["summary_row_height"] * len(case_order) * 1.6 + FIG["summary_extra_height"],
    ),
    sharex=True,
)

im_bias = axes[0].imshow(bias_grid.values, aspect="auto", cmap="RdBu_r", vmin=-bias_abs, vmax=bias_abs)
axes[0].set_title(f"Bias: model - {reference_name}", loc="left")
cbar_bias = fig.colorbar(im_bias, ax=axes[0], label=plot_units, fraction=0.025, pad=0.02)
cbar_bias.ax.tick_params(labelsize=FS["tick"])
cbar_bias.set_label(plot_units, fontsize=FS["colorbar"])

im_spread = axes[1].imshow(spread_grid.values, aspect="auto", cmap="YlOrRd")
axes[1].set_title("Ensemble spread", loc="left")
cbar_spread = fig.colorbar(im_spread, ax=axes[1], label=plot_units, fraction=0.025, pad=0.02)
cbar_spread.ax.tick_params(labelsize=FS["tick"])
cbar_spread.set_label(plot_units, fontsize=FS["colorbar"])

im_rmse = axes[2].imshow(rmse_grid.values, aspect="auto", cmap="YlOrRd")
axes[2].set_title("RMSE", loc="left")
cbar_rmse = fig.colorbar(im_rmse, ax=axes[2], label=plot_units, fraction=0.025, pad=0.02)
cbar_rmse.ax.tick_params(labelsize=FS["tick"])
cbar_rmse.set_label(plot_units, fontsize=FS["colorbar"])

im_acc = axes[3].imshow(acc_grid.values, aspect="auto", cmap="YlGn", vmin=-1, vmax=1)
axes[3].set_title("ACC", loc="left")
cbar_acc = fig.colorbar(im_acc, ax=axes[3], label="correlation", fraction=0.025, pad=0.02)
cbar_acc.ax.tick_params(labelsize=FS["tick"])
cbar_acc.set_label("correlation", fontsize=FS["colorbar"])

for ax in axes:
    ax.set_yticks(np.arange(len(case_order)))
    ax.set_yticklabels(case_order)
    ax.set_ylabel("Init / experiment")
    ax.set_xticks(heatmap_tick_positions)
    ax.set_xticklabels(heatmap_tick_labels, rotation=0, fontsize=FS["tick"])
    ax.tick_params(axis="y", labelsize=FS["tick"])
    ax.set_xticks(np.arange(-0.5, len(lead_order), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(case_order), 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=FIG["thin_line_width"])
    ax.tick_params(which="minor", bottom=False, left=False)
axes[-1].set_xlabel("Lead time (months since initialization)")

fig.suptitle(f"{region_name} {plot_name} numeric summary", fontsize=FS["suptitle"], y=1.01)
fig.tight_layout()

figname = figure_filename(field, "drift_skill_summary", reference_name, region_name)
figpath = FIGURE_OUTDIR / figname
mov.save_figure(
    fig, figpath,
    mode="",
    metric="drift",
    title=f"Drift Analysis: {field}",
    caption=f"{reference_name} reference, {region_name}",
    dpi=FIG["dpi"],
)
print(figpath)
plt.show()



def build_summary_df_from_diagnostics(field_diagnostics):
    rows = []
    for exp_name in experiments:
        for init_month in init_months:
            diag = field_diagnostics[exp_name][init_month]
            bias = diag["bias"]
            spread = diag["model_spread"]
            rmse = diag["rmse"]
            acc = diag["acc"]
            for L in bias.L.values:
                rows.append(
                    {
                        "experiment": exp_name,
                        "init_month": int(init_month),
                        "lead": int(L),
                        "lead_month": int(L) - 1,
                        "calendar_month": month_names[lead_to_cal_month(init_month, L) - 1],
                        "bias": float(bias.sel(L=L)),
                        "spread": float(spread.sel(L=L)),
                        "rmse": float(rmse.sel(L=L)),
                        "acc": float(acc.sel(L=L)),
                        "init_label": init_labels.get(init_month, f"{init_month:02d} init"),
                    }
                )
    out = pd.DataFrame(rows)
    out["case"] = out["init_label"] + " | " + out["experiment"]
    return out


def plot_summary_heatmap(summary_df_in, field_key, field_name, field_units):
    local_lead_order = sorted(summary_df_in["lead"].unique())
    local_case_order = [
        f"{init_labels.get(init_month, f'{init_month:02d} init')} | {exp_name}"
        for init_month in init_months
        for exp_name in experiments
    ]
    tick_idx = list(range(0, len(local_lead_order), lead_label_every))
    if tick_idx and tick_idx[-1] != len(local_lead_order) - 1:
        tick_idx.append(len(local_lead_order) - 1)
    tick_positions = np.asarray(tick_idx)
    tick_labels = [f"L{int(local_lead_order[i]) - 1}" for i in tick_idx]

    bias_grid = summary_df_in.pivot(index="case", columns="lead", values="bias").reindex(index=local_case_order, columns=local_lead_order)
    spread_grid = summary_df_in.pivot(index="case", columns="lead", values="spread").reindex(index=local_case_order, columns=local_lead_order)
    rmse_grid = summary_df_in.pivot(index="case", columns="lead", values="rmse").reindex(index=local_case_order, columns=local_lead_order)
    acc_grid = summary_df_in.pivot(index="case", columns="lead", values="acc").reindex(index=local_case_order, columns=local_lead_order)

    local_bias_abs = max(abs(summary_df_in["bias"].min()), abs(summary_df_in["bias"].max()))
    fig, axes = plt.subplots(
        4,
        1,
        figsize=(
            max(FIG["summary_min_width"], FIG["summary_lead_width"] * len(local_lead_order)),
            FIG["summary_row_height"] * len(local_case_order) * 1.6 + FIG["summary_extra_height"],
        ),
        sharex=True,
    )

    images = [
        axes[0].imshow(bias_grid.values, aspect="auto", cmap="RdBu_r", vmin=-local_bias_abs, vmax=local_bias_abs),
        axes[1].imshow(spread_grid.values, aspect="auto", cmap="YlOrRd"),
        axes[2].imshow(rmse_grid.values, aspect="auto", cmap="YlOrRd"),
        axes[3].imshow(acc_grid.values, aspect="auto", cmap="YlGn", vmin=-1, vmax=1),
    ]
    titles = [f"Bias: model - {reference_name}", "Ensemble spread", "RMSE", "ACC"]
    cbar_labels = [field_units, field_units, field_units, "correlation"]
    for ax, image, title, cbar_label in zip(axes, images, titles, cbar_labels):
        ax.set_title(title, loc="left")
        cbar = fig.colorbar(image, ax=ax, label=cbar_label, fraction=0.025, pad=0.02)
        cbar.ax.tick_params(labelsize=FS["tick"])
        cbar.set_label(cbar_label, fontsize=FS["colorbar"])
        ax.set_yticks(np.arange(len(local_case_order)))
        ax.set_yticklabels(local_case_order)
        ax.set_ylabel("Init / experiment")
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels, rotation=0, fontsize=FS["tick"])
        ax.tick_params(axis="y", labelsize=FS["tick"])
        ax.set_xticks(np.arange(-0.5, len(local_lead_order), 1), minor=True)
        ax.set_yticks(np.arange(-0.5, len(local_case_order), 1), minor=True)
        ax.grid(which="minor", color="white", linewidth=FIG["thin_line_width"])
        ax.tick_params(which="minor", bottom=False, left=False)
    axes[-1].set_xlabel("Lead time (months since initialization)")
    fig.suptitle(f"{region_name} {field_name} numeric summary", fontsize=FS["suptitle"], y=1.01)
    fig.tight_layout()
    figpath = FIGURE_OUTDIR / figure_filename(field_key, "drift_skill_summary", reference_name, region_name)
    mov.save_figure(
        fig, figpath,
        mode="",
        metric="drift",
        title=f"Drift Analysis: {field}",
        caption=f"{reference_name} reference, {region_name}",
        dpi=FIG["dpi"],
    )
    print(figpath)
    plt.show()


if "surface_diagnostics_by_field" in globals():
    for companion_field in summary_heatmap_companion_fields:
        if companion_field == field:
            continue
        if companion_field not in surface_diagnostics_by_field:
            print(f"Skipping {companion_field} summary heatmap: diagnostics not available.")
            continue
        cfg = VARIABLE_CONFIG[companion_field]
        companion_summary_df = build_summary_df_from_diagnostics(surface_diagnostics_by_field[companion_field])
        plot_summary_heatmap(companion_summary_df, companion_field, cfg["plot_name"], cfg["plot_units"])


## Visual Quick-Look Figures


In [ ]:
# Compact visual checks for initialization shock, drift, spread, and skill.
# These figures complement the tables above and are meant for quick experiment screening.


def build_case_summary_from_diagnostics(field_diagnostics):
    rows = []
    for exp_name in experiments:
        for init_month in init_months:
            diag = field_diagnostics[exp_name][init_month]
            bias = diag["bias"]
            rows.append(
                {
                    "experiment": exp_name,
                    "init_month": int(init_month),
                    "init_label": init_labels.get(init_month, f"{init_month:02d} init"),
                    "early_shock": diag["shock_index"],
                    "early_shock_season1": diag["shock_index_season1"],
                    "month_1_bias": float(bias.sel(L=1)),
                    "season_1_drift": season1_drift_from_bias(bias, shock_leads_season1),
                    "spread": safe_sel_mean(diag["model_spread"], skill_leads),
                    "RMSE": safe_sel_mean(diag["rmse"], skill_leads),
                    "ACC": safe_sel_mean(diag["acc"], skill_leads),
                }
            )
    out = pd.DataFrame(rows)
    out["case"] = out["init_label"] + " | " + out["experiment"]
    return out


def get_final_plot_diagnostics(field_key):
    if field_key == field:
        return diagnostics
    if "surface_diagnostics_by_field" in globals() and field_key in surface_diagnostics_by_field:
        return surface_diagnostics_by_field[field_key]
    return None


def plot_initialization_quicklook(field_key, field_diagnostics):
    cfg = VARIABLE_CONFIG[field_key]
    field_name = cfg["plot_name"]
    field_units = cfg["plot_units"]
    case_summary_plot = build_case_summary_from_diagnostics(field_diagnostics)
    case_order = [
        f"{init_labels.get(init_month, f'{init_month:02d} init')} | {exp_name}"
        for init_month in init_months
        for exp_name in experiments
    ]
    case_summary_plot = case_summary_plot.set_index("case").reindex(case_order).reset_index()
    case_labels = case_summary_plot["case"].tolist()
    case_y = np.arange(len(case_summary_plot))
    marker_cycle = ["o", "s", "^", "D", "P", "X"]
    marker_by_exp = {
        exp_name: experiment_markers.get(exp_name, marker_cycle[i % len(marker_cycle)])
        for i, exp_name in enumerate(experiments)
    }

    metric_specs = [
        ("early_shock", f"Early shock L1-L3\n({field_units})", "{:.2f}"),
        ("month_1_bias", f"Month-1 bias\n({field_units})", "{:+.2f}"),
        ("season_1_drift", f"Season-1 drift\n({field_units}/lead)", "{:+.2f}"),
        ("spread", f"Mean spread L1-L6\n({field_units})", "{:.2f}"),
        ("RMSE", f"Mean RMSE L1-L6\n({field_units})", "{:.2f}"),
        ("ACC", "Mean ACC L1-L6", "{:.2f}"),
    ]

    fig, axes = plt.subplots(
        3,
        2,
        figsize=(15.5, 11.0),
        sharey=True,
        constrained_layout=True,
    )
    axes = axes.ravel()
    for panel_index, (ax, (metric, title, fmt)) in enumerate(zip(axes, metric_specs)):
        values = case_summary_plot[metric].astype(float).values
        finite = values[np.isfinite(values)]
        if finite.size == 0:
            finite = np.asarray([0.0])
        vmin = min(0.0, float(finite.min()))
        vmax = max(0.0, float(finite.max()))
        if metric == "ACC":
            vmin = min(vmin, -0.05)
            vmax = max(vmax, 1.0)
        span = vmax - vmin if vmax != vmin else max(abs(vmax), 1.0)
        pad = max(0.14 * span, quicklook_bar_label_inside_pad)
        ax.set_xlim(vmin - pad, vmax + pad * 1.45)
        ax.axvline(0, color="0.35", linewidth=FIG["thin_line_width"], linestyle=":")

        for y, (_, row) in zip(case_y, case_summary_plot.iterrows()):
            value = float(row[metric])
            if not np.isfinite(value):
                continue
            exp_name = row["experiment"]
            color = exp_colors.get(exp_name, "0.4")
            ax.hlines(y, 0, value, color=color, alpha=0.35, linewidth=2.2)
            ax.scatter(
                value,
                y,
                s=95,
                marker=marker_by_exp[exp_name],
                color=color,
                edgecolor="white",
                linewidth=0.9,
                zorder=3,
            )
            offset = 7 if value >= 0 else -7
            ax.annotate(
                fmt.format(value),
                (value, y),
                xytext=(offset, 0),
                textcoords="offset points",
                va="center",
                ha="left" if value >= 0 else "right",
                fontsize=FS["tick"],
            )

        ax.set_title(title, loc="left", fontsize=FS["title"])
        ax.set_yticks(case_y)
        if panel_index % 2 == 0:
            ax.set_yticklabels(case_labels, fontsize=FS["tick"])
        else:
            ax.tick_params(axis="y", labelleft=False)
        ax.invert_yaxis()
        ax.grid(True, axis="x", alpha=FIG["grid_alpha"])
        ax.tick_params(axis="x", labelsize=FS["tick"])
        ax.spines[["top", "right"]].set_visible(False)

    legend_handles = [
        Line2D(
            [0],
            [0],
            marker=marker_by_exp[exp_name],
            linestyle="",
            markerfacecolor=exp_colors.get(exp_name, "0.4"),
            markeredgecolor="white",
            markersize=9,
            label=exp_name,
        )
        for exp_name in experiments
    ]
    fig.legend(handles=legend_handles, loc="upper center", ncol=len(legend_handles), frameon=False, bbox_to_anchor=(0.5, 1.025))
    fig.suptitle(f"{region_name} {field_name}: initialization quick-look metrics", fontsize=FS["suptitle"])
    figpath = FIGURE_OUTDIR / figure_filename(field_key, "quicklook_dashboard", reference_name, region_name)
    mov.save_figure(
        fig, figpath,
        mode="",
        metric="drift",
        title=f"Drift Analysis: {field}",
        caption=f"{reference_name} reference, {region_name}",
        dpi=FIG["dpi"],
    )
    print(figpath)
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.8), sharex=True)
    for exp_name in experiments:
        sub = case_summary_plot[case_summary_plot["experiment"].eq(exp_name)]
        axes[0].scatter(sub["early_shock"], sub["RMSE"], s=90, color=exp_colors.get(exp_name), label=exp_name, edgecolor="white", linewidth=0.8)
        axes[1].scatter(sub["early_shock"], sub["ACC"], s=90, color=exp_colors.get(exp_name), label=exp_name, edgecolor="white", linewidth=0.8)
        for _, row in sub.iterrows():
            label = row["init_label"].split()[0]
            axes[0].annotate(label, (row["early_shock"], row["RMSE"]), xytext=(5, 5), textcoords="offset points", fontsize=FS["tick"])
            axes[1].annotate(label, (row["early_shock"], row["ACC"]), xytext=(5, 5), textcoords="offset points", fontsize=FS["tick"])

    axes[0].set_ylabel(f"Mean RMSE L1-L6 ({field_units})")
    axes[1].set_ylabel("Mean ACC L1-L6")
    for ax in axes:
        ax.set_xlabel(f"Early shock L1-L3 ({field_units})")
        apply_axis_style(ax)
        ax.legend(frameon=False, fontsize=FS["legend"])
    axes[0].set_title("Shock vs error", loc="left")
    axes[1].set_title("Shock vs correlation", loc="left")
    fig.suptitle(f"{region_name} {field_name}: shock-skill relationship", fontsize=FS["suptitle"], y=1.03)
    fig.tight_layout()
    figpath = FIGURE_OUTDIR / figure_filename(field_key, "shock_skill_scatter", reference_name, region_name)
    mov.save_figure(
        fig, figpath,
        mode="",
        metric="drift",
        title=f"Drift Analysis: {field}",
        caption=f"{reference_name} reference, {region_name}",
        dpi=FIG["dpi"],
    )
    print(figpath)
    plt.show()


def plot_final_leadtime_diagnostics(field_key, field_diagnostics):
    cfg = VARIABLE_CONFIG[field_key]
    field_name = cfg["plot_name"]
    field_units = cfg["plot_units"]
    summary_df_in = build_summary_df_from_diagnostics(field_diagnostics)
    lead_order_in = sorted(summary_df_in["lead"].unique())
    lead_metric_specs = [
        ("bias", f"Bias ({field_units})", True),
        ("rmse", f"RMSE ({field_units})", False),
        ("acc", "ACC", True),
        ("spread", f"Spread ({field_units})", False),
    ]
    fig, axes = plt.subplots(len(lead_metric_specs), len(init_months), figsize=(7.0 * len(init_months), 11.5), sharex=True)
    if len(init_months) == 1:
        axes = np.asarray(axes).reshape(len(lead_metric_specs), 1)

    for col, init_month in enumerate(init_months):
        init_label = init_labels.get(init_month, f"{init_month:02d} init")
        for row, (metric, ylabel, draw_zero) in enumerate(lead_metric_specs):
            ax = axes[row, col]
            for exp_name in experiments:
                sub = summary_df_in[(summary_df_in["experiment"].eq(exp_name)) & (summary_df_in["init_month"].eq(init_month))].sort_values("lead")
                ax.plot(sub["lead"] - 1, sub[metric], marker="o", linewidth=FIG["line_width"], markersize=FIG["marker_size"], color=exp_colors.get(exp_name), label=exp_name)
            if draw_zero:
                ax.axhline(0, color="0.25", linewidth=FIG["thin_line_width"])
            if row == 0:
                ax.set_title(init_label, loc="left")
            if col == 0:
                ax.set_ylabel(ylabel)
            apply_axis_style(ax)
            format_lead_time_axis(ax, np.asarray(lead_order_in), init_month, show_labels=(row == len(lead_metric_specs) - 1))
            if row == len(lead_metric_specs) - 1:
                ax.set_xlabel("Lead time and target month")
            if row == 0 and col == len(init_months) - 1:
                ax.legend(frameon=False, fontsize=FS["legend"])

    fig.suptitle(f"{region_name} {field_name}: lead-time diagnostics", fontsize=FS["suptitle"], y=1.01)
    fig.tight_layout()
    figpath = FIGURE_OUTDIR / figure_filename(field_key, "leadtime_diagnostics", reference_name, region_name)
    mov.save_figure(
        fig, figpath,
        mode="",
        metric="drift",
        title=f"Drift Analysis: {field}",
        caption=f"{reference_name} reference, {region_name}",
        dpi=FIG["dpi"],
    )
    print(figpath)
    plt.show()


for plot_field in final_plot_fields:
    field_diagnostics = get_final_plot_diagnostics(plot_field)
    if field_diagnostics is None:
        print(f"Skipping final plots for {plot_field}: diagnostics not available.")
        continue
    plot_initialization_quicklook(plot_field, field_diagnostics)
    plot_final_leadtime_diagnostics(plot_field, field_diagnostics)


In [ ]:
# -----------------------------
# Clean up Dask resources when finished.
# -----------------------------
# close_cluster(cluster, client)